# Q-learning in a maze
**Machine Learning · Week 7 (Reinforcement learning) · Day 4**

Lesson page: https://sara-aminpour.com/tutorials/ml/w7d4.html

A cat starts in a corner of a small grid and wants the fish. Each step costs a little; a puddle hurts.
With **Q-learning** it learns, by trial and error, how good every move is from every square.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

maze = [
    "S...#.",
    ".##.#.",
    "...P..",
    ".#.##.",
    ".#....",
    "...#.F",
]
rows, cols = len(maze), len(maze[0])
moves = {0: (-1, 0), 1: (0, 1), 2: (1, 0), 3: (0, -1)}           # up, right, down, left
arrows = "↑→↓←"

def step(state, action):
    r, c = state
    dr, dc = moves[action]
    nr, nc = r + dr, c + dc
    if not (0 <= nr < rows and 0 <= nc < cols) or maze[nr][nc] == "#":
        nr, nc = r, c                                             # bumped into a wall: stay put
    cell = maze[nr][nc]
    if cell == "F":
        return (nr, nc), 1.0, True                                # the fish: big reward, episode ends
    if cell == "P":
        return (nr, nc), -1.0, False                              # a puddle: ouch
    return (nr, nc), -0.04, False                                 # every step costs a little

print("\n".join(maze))

## The Q-learning update
$Q(s, a) \leftarrow Q(s, a) + \alpha\,\big[r + \gamma \max_{a'} Q(s', a') - Q(s, a)\big]$:
nudge the old guess towards "reward now + discounted value of where I landed".

In [ ]:
def q_learning(episodes=400, alpha=0.5, gamma=0.95, epsilon=0.2, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((rows, cols, 4))
    lengths = []
    for episode in range(episodes):
        state, done, n = (0, 0), False, 0
        while not done and n < 200:
            if rng.random() < epsilon:
                action = int(rng.integers(4))                     # explore
            else:
                action = int(np.argmax(Q[state]))                 # exploit
            new_state, reward, done = step(state, action)
            target = reward + (0 if done else gamma * Q[new_state].max())
            Q[state][action] += alpha * (target - Q[state][action])
            state, n = new_state, n + 1
        lengths.append(n)
    return Q, lengths

Q, lengths = q_learning()
print("steps in the first 5 episodes:", lengths[:5])
print("steps in the last 5 episodes :", lengths[-5:])

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
values = Q.max(axis=2)
for r in range(rows):
    for c in range(cols):
        if maze[r][c] == "#":
            values[r, c] = np.nan
cmap = plt.get_cmap("Purples").copy()
cmap.set_bad("#3a3456")                                   # walls drawn dark
shown = left.imshow(np.ma.masked_invalid(values), cmap=cmap)
fig.colorbar(shown, ax=left, label="best Q value")
for r in range(rows):
    for c in range(cols):
        cell = maze[r][c]
        text = {"#": "", "F": "fish", "P": "puddle"}.get(cell, arrows[int(np.argmax(Q[r, c]))])
        color = "white" if np.nan_to_num(values[r, c]) > 0.55 else "#2c2842"
        left.text(c, r, text, ha="center", va="center", fontsize=13 if len(text) == 1 else 8, color=color)
left.set_xticks([])
left.set_yticks([])
left.grid(False)
left.set_title("What the cat learned: values and best moves")
right.plot(lengths, color="#ff8a50", lw=1)
right.set_xlabel("episode")
right.set_ylabel("steps to reach the fish")
right.set_title("Learning curve")
plt.show()

## Check against value iteration
Yesterday's value iteration computed the values from the rules. Q-learning never saw the rules,
only its own moves. Do they agree?

In [ ]:
V = np.zeros((rows, cols))
for sweep in range(60):
    new_V = V.copy()
    for r in range(rows):
        for c in range(cols):
            if maze[r][c] in "#F":
                continue
            outcomes = []
            for a in range(4):
                (nr, nc), reward, done = step((r, c), a)
                outcomes.append(reward + (0 if done else 0.95 * V[nr, nc]))
            new_V[r, c] = max(outcomes)
    V = new_V
print("value of the start square, value iteration:", round(V[0, 0], 3))
print("value of the start square, Q-learning     :", round(Q[0, 0].max(), 3))

## Your turn: explore less as you learn more
A fixed ε = 0.2 keeps taking random steps forever, even after the cat knows the way. Write
`q_learning_decay`, a copy of `q_learning` where ε starts at 1 (pure exploration) and shrinks every
episode: `epsilon = max(0.01, 0.99 ** episode)`. Compare the average episode length over the last 50
episodes with the fixed ε = 0.2 (the shortest possible route takes 10 steps).

In [ ]:
def q_learning_decay(episodes=400, alpha=0.5, gamma=0.95, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((rows, cols, 4))
    lengths = []
    for episode in range(episodes):
        epsilon = 0.2                                             # TODO: max(0.01, 0.99 ** episode)
        state, done, n = (0, 0), False, 0
        while not done and n < 200:
            action = int(rng.integers(4)) if rng.random() < epsilon else int(np.argmax(Q[state]))
            new_state, reward, done = step(state, action)
            target = reward + (0 if done else gamma * Q[new_state].max())
            Q[state][action] += alpha * (target - Q[state][action])
            state, n = new_state, n + 1
        lengths.append(n)
    return Q, lengths

print("fixed epsilon  :", np.mean(q_learning()[1][-50:]))
print("decaying epsilon:", np.mean(q_learning_decay()[1][-50:]))

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def q_learning_decay(episodes=400, alpha=0.5, gamma=0.95, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((rows, cols, 4))
    lengths = []
    for episode in range(episodes):
        epsilon = max(0.01, 0.99 ** episode)                      # explore a lot early, very little later
        state, done, n = (0, 0), False, 0
        while not done and n < 200:
            action = int(rng.integers(4)) if rng.random() < epsilon else int(np.argmax(Q[state]))
            new_state, reward, done = step(state, action)
            target = reward + (0 if done else gamma * Q[new_state].max())
            Q[state][action] += alpha * (target - Q[state][action])
            state, n = new_state, n + 1
        lengths.append(n)
    return Q, lengths

for seed in range(3):
    fixed, decay = q_learning(seed=seed)[1], q_learning_decay(seed=seed)[1]
    print(f"seed {seed}: last 50 episodes, average steps: fixed epsilon {np.mean(fixed[-50:]):.1f}, "
          f"decaying epsilon {np.mean(decay[-50:]):.1f}")
print("Exploring a lot at first and very little at the end gets close to the 10-step route.")